# Fase 2 — Exploración de la estructura de la YouTube Data API v3

**Objetivo:** antes de diseñar la base de datos, entender exactamente qué forma tienen los JSON que devuelve la API para `@PuroBalompie`: qué campos hay, cuáles están anidados, qué tipos de dato trae realmente cada uno (aunque "parezcan" números) y qué necesitaremos transformar.

**Requisito previo:** haber ejecutado `python -m src.fetch_purobalompie` desde la raíz del proyecto, para que existan los archivos en `/raw`:
- `purobalompie_channel.json`
- `purobalompie_video_ids.json`
- `purobalompie_videos_detail.json`

In [1]:
import json
import os
import re

import pandas as pd

RAW_DIR = os.path.join("..", "raw")

with open(os.path.join(RAW_DIR, "purobalompie_channel.json"), encoding="utf-8") as f:
    channel = json.load(f)

with open(os.path.join(RAW_DIR, "purobalompie_videos_detail.json"), encoding="utf-8") as f:
    videos = json.load(f)

print(f"Vídeos cargados: {len(videos)}")

Vídeos cargados: 339


## 1. Estructura cruda del canal

Primero miramos las claves de primer nivel y de cada bloque (`snippet`, `statistics`, `contentDetails`) tal cual las devuelve la API, sin transformar nada todavía.

In [2]:
print("Claves de primer nivel:", list(channel.keys()))
print("Claves de snippet:", list(channel["snippet"].keys()))
print("Claves de statistics:", list(channel["statistics"].keys()))
print("Claves de contentDetails:", list(channel["contentDetails"].keys()))

Claves de primer nivel: ['kind', 'etag', 'id', 'snippet', 'contentDetails', 'statistics']
Claves de snippet: ['title', 'description', 'customUrl', 'publishedAt', 'thumbnails', 'localized', 'country']
Claves de statistics: ['viewCount', 'subscriberCount', 'hiddenSubscriberCount', 'videoCount']
Claves de contentDetails: ['relatedPlaylists']


## 2. Aplanar el canal con `pandas.json_normalize`

`json_normalize` convierte los diccionarios anidados en columnas del tipo `snippet.title`, `statistics.viewCount`, etc. Esto nos da de un vistazo **todas** las columnas reales que tendríamos disponibles para la tabla "canal" de la base de datos.

In [3]:
df_channel = pd.json_normalize(channel)
print(f"Total columnas: {len(df_channel.columns)}\n")
for col in df_channel.columns:
    valor = df_channel[col].iloc[0]
    print(f"{col:45s} -> tipo Python: {type(valor).__name__:10s} | valor: {valor}")

Total columnas: 25

kind                                          -> tipo Python: str        | valor: youtube#channel
etag                                          -> tipo Python: str        | valor: xb0YI05B4pDHNPkwSJ2WWMEbCNo
id                                            -> tipo Python: str        | valor: UCXO-s_j387f-5Ta_JzGOQJA
snippet.title                                 -> tipo Python: str        | valor: Puro Balompié
snippet.description                           -> tipo Python: str        | valor: Mucho fútbol. Mucho Betis. Puro balompié.
snippet.customUrl                             -> tipo Python: str        | valor: @purobalompie
snippet.publishedAt                           -> tipo Python: str        | valor: 2015-04-11T17:19:51Z
snippet.thumbnails.default.url                -> tipo Python: str        | valor: https://yt3.ggpht.com/iXSvl8XNoxoWwEXW-cD8fEzRFDkbqHsmpGBWg77w-U4xLGqypQqYOL4B6_QJx-67sGGkQNO1=s88-c-k-c0x00ffffff-no-rj
snippet.thumbnails.default.width           

## 3. Aplanar el detalle de vídeos

Repetimos el proceso con la lista de vídeos. Al ser una lista de diccionarios, `json_normalize` nos da directamente un DataFrame con una fila por vídeo.

In [4]:
df_videos = pd.json_normalize(videos)
print(f"Total vídeos: {len(df_videos)} | Total columnas: {len(df_videos.columns)}\n")
print(df_videos.dtypes)

Total vídeos: 339 | Total columnas: 43

kind                                      str
etag                                      str
id                                        str
snippet.publishedAt                       str
snippet.channelId                         str
snippet.title                             str
snippet.description                       str
snippet.thumbnails.default.url            str
snippet.thumbnails.default.width        int64
snippet.thumbnails.default.height       int64
snippet.thumbnails.medium.url             str
snippet.thumbnails.medium.width         int64
snippet.thumbnails.medium.height        int64
snippet.thumbnails.high.url               str
snippet.thumbnails.high.width           int64
snippet.thumbnails.high.height          int64
snippet.thumbnails.standard.url           str
snippet.thumbnails.standard.width       int64
snippet.thumbnails.standard.height      int64
snippet.thumbnails.maxres.url             str
snippet.thumbnails.maxres.width       fl

In [5]:
# Vista rápida de las columnas que más nos interesan para el análisis del cliente
columnas_clave = [
    "id",
    "snippet.title",
    "snippet.publishedAt",
    "snippet.tags",
    "snippet.categoryId",
    "statistics.viewCount",
    "statistics.likeCount",
    "statistics.commentCount",
    "contentDetails.duration",
]
df_videos[[c for c in columnas_clave if c in df_videos.columns]].head(10)

,id,snippet.title,snippet.publishedAt,snippet.tags,snippet.categoryId,statistics.viewCount,statistics.likeCount,statistics.commentCount,contentDetails.duration
0,aIMLDjqFVSQ,El Netflix de las Camisetas de Fútbol (solució...,2026-09-11T15:15:10Z,"[BETIS, REAL BETIS, BETICISMO, BÉTICO, REAL BE...",17,36304,502,126,PT9M5S
1,tDkEoTVHN3M,Hablemos del debut en CHAMPIONS y TROY PARROTT,2026-09-09T09:08:08Z,"[BETIS, REAL BETIS, BETICISMO, BÉTICO, REAL BE...",17,16136,590,182,PT14M19S
2,nV11a3pNzoQ,TODO lo que hemos vivido para volver a la Cham...,2026-09-07T13:20:35Z,"[BETIS, REAL BETIS, BETICISMO, BÉTICO, REAL BE...",17,28860,385,72,PT8M48S
3,H8KKyXKq9h8,LILLE - BETIS: así es el primer rival europeo ...,2026-09-06T16:00:09Z,"[BETIS, REAL BETIS, BETICISMO, BÉTICO, REAL BE...",17,15003,405,72,PT17M35S
4,_llcu2dHHNI,Parrot VS Real Madrid,2026-09-05T11:15:49Z,"[BETIS, REAL BETIS, BETICISMO, BÉTICO, REAL BE...",17,25239,857,62,PT1M21S
5,ZZSUL31nhIs,Me invitan a hablar del BETIS–MADRID… y digo esto,2026-09-04T14:50:25Z,"[BETIS, REAL BETIS, BETICISMO, BÉTICO, REAL BE...",17,8167,254,83,PT25M2S
6,NNNld_kME8o,Lo que ilusiona (MUCHO) de este BETIS,2026-09-03T12:16:13Z,"[BETIS, REAL BETIS, BETICISMO, BÉTICO, REAL BE...",17,11117,381,126,PT11M5S
7,m_-3GrxDUIc,BETIS: ¿Qué nota le pones al mercado de fichajes?,2026-09-01T23:19:37Z,"[BETIS, REAL BETIS, BETICISMO, BÉTICO, REAL BE...",17,14462,458,276,PT1H2M34S
8,VvIoEh2yPPk,"YONKIS DE LOS FICHAJES, ¿quiénes son?",2026-09-01T14:03:38Z,"[BETIS, REAL BETIS, BETICISMO, BÉTICO, REAL BE...",17,4450,146,54,PT6M35S
9,2Mil73s8fW4,Últimas 24 horas: Así se moverá el BETIS en el...,2026-08-31T11:44:48Z,"[BETIS, REAL BETIS, BETICISMO, BÉTICO, REAL BE...",17,19151,503,147,PT8M1S


## 4. Hallazgos importantes antes de diseñar la base de datos

Al inspeccionar los tipos reales (no lo que "parecen" a simple vista), aparecen varios puntos a los que hay que prestar atención:

1. **`viewCount`, `likeCount`, `commentCount`, `subscriberCount` llegan como texto (`str`)**, no como número, aunque su contenido sea numérico. Habrá que convertirlos explícitamente a `int` antes de cualquier cálculo o carga a base de datos.
2. **`contentDetails.duration` viene en formato ISO 8601** (p. ej. `PT14M32S` = 14 minutos y 32 segundos), no en segundos ni en `HH:MM:SS`. Hay que parsearlo.
3. **`snippet.thumbnails` es un diccionario de resoluciones** (`default`, `medium`, `high`...), cada una con `url`, `width` y `height`: es una estructura anidada de dos niveles que se aplana en varias columnas.
4. **`snippet.tags` es una lista de strings** de longitud variable por vídeo — no encaja como columna plana de una tabla relacional; probablemente merezca su propia tabla (`video_tags`) en el modelo relacional.
5. **`hiddenSubscriberCount`** es un booleano a vigilar: si es `true`, el número de suscriptores públicos puede no ser fiable/visible.

Estas transformaciones (tipos, duración, tags) son justo el trabajo de limpieza que documentaremos como paso previo al diseño del modelo de base de datos.

## 5. Función de apoyo: convertir duración ISO 8601 a segundos

Pequeña utilidad (sin dependencias externas) para transformar `PT14M32S` en segundos totales, que es el formato que necesitaremos para calcular duración media, comparar con la retención, etc.

In [8]:
def duracion_iso8601_a_segundos(duracion: str) -> int:
    """Convierte 'PT1H2M10S' -> segundos totales. Soporta horas, minutos y segundos."""
    patron = re.compile(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?")
    match = patron.match(duracion)
    horas, minutos, segundos = (int(x) if x else 0 for x in match.groups())
    return horas * 3600 + minutos * 60 + segundos


if "contentDetails.duration" in df_videos.columns:
    df_videos["duracion_segundos"] = df_videos["contentDetails.duration"].apply(
        duracion_iso8601_a_segundos
    )

for col in ["statistics.viewCount", "statistics.likeCount", "statistics.commentCount"]:
    if col in df_videos.columns:
        df_videos[col] = pd.to_numeric(df_videos[col], errors="coerce")

df_videos[["snippet.title", "duracion_segundos", "statistics.viewCount"]].head(10)

,snippet.title,duracion_segundos,statistics.viewCount
0,El Netflix de las Camisetas de Fútbol (solució...,545,36304
1,Hablemos del debut en CHAMPIONS y TROY PARROTT,859,16136
2,TODO lo que hemos vivido para volver a la Cham...,528,28860
3,LILLE - BETIS: así es el primer rival europeo ...,1055,15003
4,Parrot VS Real Madrid,81,25239
5,Me invitan a hablar del BETIS–MADRID… y digo esto,1502,8167
6,Lo que ilusiona (MUCHO) de este BETIS,665,11117
7,BETIS: ¿Qué nota le pones al mercado de fichajes?,3754,14462
8,"YONKIS DE LOS FICHAJES, ¿quiénes son?",395,4450
9,Últimas 24 horas: Así se moverá el BETIS en el...,481,19151


## Próximo paso

Con esta exploración ya sabemos exactamente qué trae la Data API v3 y en qué formato. El siguiente paso es documentar de la misma forma la estructura que **devolvería** la Analytics API (retención, audiencia, ingresos) para poder generar datos sintéticos con el mismo formato de columnas, y así unificar ambas fuentes antes de diseñar el modelo de base de datos.